In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

### Load Teacher Model

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location='cpu')
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
teacher_model.to(device)

device

device(type='cuda', index=1)

In [28]:
from train_eval_func import eval_loop

In [29]:
eval_loop(teacher_model, val_dataloader, task_name, device)

({'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624},
 0.8802372308877798)

# 5. Layer Score Importance And Sorting

In [30]:
from structure_prune_helper import compute_layer_importance_scores, register_importance_masks, remove_importance_masks

In [31]:
register_importance_masks(teacher_model, device, register_mha_layer=True, register_mlp_layer=True,
                         register_mha=True, register_mlp=True)

  Layer 0: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 1: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 2: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 3: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 4: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 5: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 6: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 7: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 8: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 9: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 10: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 11: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [32]:
from layer_merge_helper import reorder_merged_layer_heads

In [33]:
mha_imp, ffn_imp, head_imp, neuron_imp, sublayer_imp = compute_layer_importance_scores(
    teacher_model, train_dataloader, device,
    compute_mha=True, compute_ffn=True,
    compute_heads=True, compute_neurons=True
)

Computing importance (MHA layer, FFN layer, heads, neurons): 100%|██████████| 115/115 [00:11<00:00,  9.61it/s]


Layer Importance Summary (separate):
          Layer     MHA(layer)     FFN(layer)    Heads(mean)  Neurons(mean)
  ---------------------------------------------------------------------------
  0                     0.7879         0.9843         0.0000         0.0000
  1                     0.7108         0.4983         0.0000         0.0000
  2                     0.5232         0.6907         0.0000         0.0000
  3                     0.4300         0.5575         0.0000         0.0000
  4                     0.4305         1.0000         0.0000         0.0000
  5                     1.0000         0.5937         0.0000         0.0000
  6                     0.2730         0.4748         0.0000         0.0000
  7                     0.2159         0.2461         0.0000         0.0000
  8                     0.3048         0.4255         0.0000         0.0000
  9                     0.2532         0.2987         0.0000         0.0000
  10                    0.1479         0.1349   

In [34]:
for i in range(12):
    reorder_merged_layer_heads(model.bert.encoder.layer[i], head_imp[i].cpu(), torch.device("cpu"))

Reordered 12 heads, new order: [8, 5, 9, 0, 1, 11, 6, 3, 10, 4, 2, 7]
Reordered 12 heads, new order: [11, 7, 8, 6, 2, 4, 1, 10, 0, 9, 5, 3]
Reordered 12 heads, new order: [6, 11, 7, 1, 10, 3, 0, 9, 5, 2, 8, 4]
Reordered 12 heads, new order: [0, 7, 10, 3, 9, 5, 4, 8, 1, 6, 11, 2]
Reordered 12 heads, new order: [2, 3, 9, 4, 1, 5, 8, 10, 11, 7, 6, 0]
Reordered 12 heads, new order: [8, 4, 3, 5, 0, 10, 7, 11, 9, 6, 2, 1]
Reordered 12 heads, new order: [0, 8, 7, 4, 6, 5, 1, 9, 11, 10, 2, 3]
Reordered 12 heads, new order: [10, 0, 5, 9, 1, 4, 8, 2, 11, 7, 3, 6]
Reordered 12 heads, new order: [8, 1, 7, 10, 9, 3, 2, 0, 11, 4, 5, 6]
Reordered 12 heads, new order: [5, 2, 11, 10, 8, 7, 1, 0, 6, 4, 3, 9]
Reordered 12 heads, new order: [0, 5, 8, 7, 1, 10, 11, 3, 9, 2, 6, 4]
Reordered 12 heads, new order: [2, 9, 8, 4, 7, 5, 10, 11, 6, 0, 1, 3]


In [35]:
layer_scores = []
for idx in range(0, 24, 2):
    layer = int(idx / 2)
    layer_score = (sublayer_imp[idx] + sublayer_imp[idx+1]) / 2
    print(layer, layer_score)
    layer_scores.append(layer_score)

0 0.8548370079721432
1 0.5887043064120773
2 0.585039680877147
3 0.4760704818221425
4 0.6834934591444725
5 0.7780108213973873
6 0.3588173180990346
7 0.22315765120006437
8 0.35161277226755316
9 0.266488999862323
10 0.13714492726932062
11 0.11579482304911151


# 6. Calculating Similarity 

In [36]:
from CKA import CKAEvaluator

cka_eval = CKAEvaluator(device)

In [37]:
reps_similarity, atts_similarity, sub_similarity = cka_eval.subLayer_interleaved_pairwise(
    teacher_model, 
    train_dataloader, 
    device, 
    only_cls_token=True, 
    max_iter=115
)

CKA Sublayer Evaluation: 100%|██████████| 115/115 [00:32<00:00,  3.58it/s]


In [38]:
reps_stats = cka_eval.similarity_stats(reps_similarity)
atts_stats = cka_eval.similarity_stats(atts_similarity)
subs_stats = cka_eval.similarity_stats(sub_similarity)

print("  Reps Similarity Stats:")
print(f"    Average: {reps_stats['average']:.6f}")
print(f"    Highest: {reps_stats['max']:.6f}")
print(f"    Lowest:  {reps_stats['min']:.6f}")
print("")

print("  Atts Similarity Stats:")
print(f"    Average: {atts_stats['average']:.6f}")
print(f"    Highest: {atts_stats['max']:.6f}")
print(f"    Lowest:  {atts_stats['min']:.6f}")

print("  Sublayer Similarity Stats:")
print(f"    Average: {subs_stats['average']:.6f}")
print(f"    Highest: {subs_stats['max']:.6f}")
print(f"    Lowest:  {subs_stats['min']:.6f}")

  Reps Similarity Stats:
    Average: 0.899861
    Highest: 0.993713
    Lowest:  0.724249

  Atts Similarity Stats:
    Average: 0.944353
    Highest: 0.991281
    Lowest:  0.763968
  Sublayer Similarity Stats:
    Average: 0.436442
    Highest: 0.724606
    Lowest:  0.141554


# 7. Creating Student Model

### Layer Composition

In [39]:
layer_composition = [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]

In [40]:
from layer_merge_helper import merge_high_sim_layer_composition

In [41]:
layer_composition = merge_high_sim_layer_composition(layer_composition, sublayer_imp, reps_similarity, layer_scores, 0.9)

In [42]:
layer_composition, len(layer_composition)

([[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]], 6)

In [43]:
print(layer_scores)

[0.8548370079721432, 0.6788142235308011, 0.6788142235308011, 0.4760704818221425, 0.6834934591444725, 0.7780108213973873, 0.7780108213973873, 0.7780108213973873, 0.35161277226755316, 0.35161277226755316, 0.35161277226755316, 0.35161277226755316]


### Merging Layers

In [44]:
def merge_mha_cluster(model, layers, target_idx, k_per_layer, device):
    assert len(layers) == len(k_per_layer), "layers and k_per_layer must be same length"
    assert 0 <= target_idx < len(layers), "target_idx out of range"

    targetLayer = layers[target_idx]
    head_size   = targetLayer.attention.self.attention_head_size

    # clamp each k to the number of heads actually available in that layer
    k_per_layer_clamped = []
    for k, layer in zip(k_per_layer, layers):
        avail = layer.attention.self.num_attention_heads
        k_clamped = min(k, avail)
        if k_clamped != k:
            print(f"  WARNING: requested k={k} but layer only has {avail} heads, clamped to {k_clamped}")
        k_per_layer_clamped.append(k_clamped)
    k_per_layer = k_per_layer_clamped

    n_head  = sum(k_per_layer)
    new_dim = n_head * head_size

    print(f"Merging cluster of {len(layers)} layers into target (idx={target_idx})")
    print(f"  k_per_layer (clamped): {k_per_layer}")
    print(f"  n_head total: {n_head}, new_dim: {new_dim}")

    for i, (layer, k) in enumerate(zip(layers, k_per_layer)):
        tag = " (target)" if i == target_idx else ""
        print(f"  Layer{tag}: taking {k} heads, head_size={layer.attention.self.attention_head_size}")

    # sanity check — all layers must have the SAME head_size
    head_sizes = set(layer.attention.self.attention_head_size for layer in layers)
    if len(head_sizes) > 1:
        raise ValueError(f"All layers must have the same head_size, got {head_sizes}")

    t_keep_len = k_per_layer[target_idx] * head_size

    orig_q   = targetLayer.attention.self.query.weight.data[:t_keep_len].clone()
    orig_q_b = targetLayer.attention.self.query.bias.data[:t_keep_len].clone()
    orig_k   = targetLayer.attention.self.key.weight.data[:t_keep_len].clone()
    orig_k_b = targetLayer.attention.self.key.bias.data[:t_keep_len].clone()
    orig_v   = targetLayer.attention.self.value.weight.data[:t_keep_len].clone()
    orig_v_b = targetLayer.attention.self.value.bias.data[:t_keep_len].clone()
    orig_o   = targetLayer.attention.output.dense.weight.data[:, :t_keep_len].clone()

    targetLayer.attention.self.query   = resize_qkv(targetLayer.attention.self.query, new_dim)
    targetLayer.attention.self.key     = resize_qkv(targetLayer.attention.self.key, new_dim)
    targetLayer.attention.self.value   = resize_qkv(targetLayer.attention.self.value, new_dim)
    targetLayer.attention.output.dense = resize_output(targetLayer.attention.output.dense, new_dim)

    with torch.no_grad():
        offset = 0

        for i, (layer, k) in enumerate(zip(layers, k_per_layer)):
            write_len = k * head_size

            # guard: catch overflow BEFORE it crashes with a confusing message
            if offset + write_len > new_dim:
                raise RuntimeError(
                    f"Offset overflow at layer {i}: offset={offset}, write_len={write_len}, "
                    f"new_dim={new_dim}. Sum of k_per_layer*head_size exceeds new_dim — "
                    f"check k_per_layer values."
                )

            if i == target_idx:
                targetLayer.attention.self.query.weight.data[offset:offset+write_len] = orig_q
                targetLayer.attention.self.query.bias.data[offset:offset+write_len]   = orig_q_b
                targetLayer.attention.self.key.weight.data[offset:offset+write_len]   = orig_k
                targetLayer.attention.self.key.bias.data[offset:offset+write_len]     = orig_k_b
                targetLayer.attention.self.value.weight.data[offset:offset+write_len] = orig_v
                targetLayer.attention.self.value.bias.data[offset:offset+write_len]   = orig_v_b
                targetLayer.attention.output.dense.weight.data[:, offset:offset+write_len] = orig_o
            else:
                # guard: source layer must actually have write_len rows available
                avail_rows = layer.attention.self.query.weight.data.shape[0]
                if write_len > avail_rows:
                    raise RuntimeError(
                        f"Layer {i}: trying to write {write_len} rows but source only has "
                        f"{avail_rows} rows (num_heads={layer.attention.self.num_attention_heads})"
                    )

                targetLayer.attention.self.query.weight.data[offset:offset+write_len] = \
                    layer.attention.self.query.weight.data[:write_len]
                targetLayer.attention.self.query.bias.data[offset:offset+write_len] = \
                    layer.attention.self.query.bias.data[:write_len]

                targetLayer.attention.self.key.weight.data[offset:offset+write_len] = \
                    layer.attention.self.key.weight.data[:write_len]
                targetLayer.attention.self.key.bias.data[offset:offset+write_len] = \
                    layer.attention.self.key.bias.data[:write_len]

                targetLayer.attention.self.value.weight.data[offset:offset+write_len] = \
                    layer.attention.self.value.weight.data[:write_len]
                targetLayer.attention.self.value.bias.data[offset:offset+write_len] = \
                    layer.attention.self.value.bias.data[:write_len]

                targetLayer.attention.output.dense.weight.data[:, offset:offset+write_len] = \
                    layer.attention.output.dense.weight.data[:, :write_len]

            print(f"    Wrote {k} heads from layer {i}{' (target)' if i==target_idx else ''} "
                  f"into slots [{offset}:{offset+write_len}]")
            offset += write_len

    targetLayer.attention.self.num_attention_heads = n_head
    targetLayer.attention.self.all_head_size        = new_dim

    targetLayer.head_mask_param = nn.Parameter(
        torch.ones(n_head, dtype=torch.float32, device=device),
        requires_grad=True
    )

    print(f"  Done: target now has {n_head} heads (dim={new_dim})")
    return targetLayer

In [45]:
import torch.nn as nn

def resize_qkv(linear_layer, new_out_dim):
    new_layer = nn.Linear(
        in_features=linear_layer.in_features,
        out_features=new_out_dim,
        bias=True
    ).to(linear_layer.weight.device)

    return new_layer

In [46]:
def resize_output(linear_layer, new_in_dim):
    new_layer = nn.Linear(
        in_features=new_in_dim,
        out_features=linear_layer.out_features,
        bias=True
    ).to(linear_layer.weight.device)

    return new_layer


In [47]:
import copy

def test_merge_mha_cluster(model, layer_indices, target_idx, k_per_layer, device):
    """
    Verifies merge_mha_cluster correctness by checking that:
    1. Target's own (kept) weights are preserved exactly
    2. Each source layer's top-k weights are copied exactly into the right offset
    3. Final shapes match expectations
    4. Non-participating layers in the model are untouched
    """
    # work on a deep copy so we don't corrupt the real model
    test_model = copy.deepcopy(model)

    layers = [test_model.bert.encoder.layer[idx] for idx in layer_indices]
    head_size = layers[0].attention.self.attention_head_size

    # --- snapshot BEFORE merge ---
    before = {}
    for i, layer in enumerate(layers):
        before[i] = {
            'q_w': layer.attention.self.query.weight.data.clone(),
            'q_b': layer.attention.self.query.bias.data.clone(),
            'k_w': layer.attention.self.key.weight.data.clone(),
            'k_b': layer.attention.self.key.bias.data.clone(),
            'v_w': layer.attention.self.value.weight.data.clone(),
            'v_b': layer.attention.self.value.bias.data.clone(),
            'o_w': layer.attention.output.dense.weight.data.clone(),
            'num_heads': layer.attention.self.num_attention_heads,
        }

    # snapshot an UNRELATED layer to confirm it's untouched
    untouched_idx = next(
        i for i in range(len(test_model.bert.encoder.layer))
        if i not in layer_indices
    )
    untouched_before = test_model.bert.encoder.layer[untouched_idx].attention.self.query.weight.data.clone()

    # --- run merge ---
    merged_layer = merge_mha_cluster(test_model, layers, target_idx, k_per_layer, device)

    # --- clamp k_per_layer same way the function does, for verification ---
    k_clamped = [min(k, before[i]['num_heads']) for i, k in enumerate(k_per_layer)]

    print("\n" + "="*60)
    print("VERIFICATION")
    print("="*60)

    all_passed = True

    # --- check 1: final shape ---
    n_head_expected = sum(k_clamped)
    actual_n_head = merged_layer.attention.self.num_attention_heads
    shape_ok = (actual_n_head == n_head_expected)
    print(f"[{'PASS' if shape_ok else 'FAIL'}] num_heads: expected={n_head_expected}, actual={actual_n_head}")
    all_passed &= shape_ok

    expected_dim = n_head_expected * head_size
    actual_dim = merged_layer.attention.self.query.weight.shape[0]
    dim_ok = (actual_dim == expected_dim)
    print(f"[{'PASS' if dim_ok else 'FAIL'}] query out_features: expected={expected_dim}, actual={actual_dim}")
    all_passed &= dim_ok

    # --- check 2: each segment matches source exactly ---
    offset = 0
    for i, k in enumerate(k_clamped):
        write_len = k * head_size
        segment_q = merged_layer.attention.self.query.weight.data[offset:offset+write_len]
        expected_q = before[i]['q_w'][:write_len]

        match = torch.allclose(segment_q, expected_q, atol=1e-7)
        tag = " (target)" if i == target_idx else ""
        print(f"[{'PASS' if match else 'FAIL'}] Layer {i}{tag} Q weights "
              f"[{offset}:{offset+write_len}] match source")
        all_passed &= match

        # also check K, V, bias for thoroughness
        segment_k = merged_layer.attention.self.key.weight.data[offset:offset+write_len]
        expected_k = before[i]['k_w'][:write_len]
        match_k = torch.allclose(segment_k, expected_k, atol=1e-7)
        print(f"[{'PASS' if match_k else 'FAIL'}] Layer {i}{tag} K weights match")
        all_passed &= match_k

        segment_v = merged_layer.attention.self.value.weight.data[offset:offset+write_len]
        expected_v = before[i]['v_w'][:write_len]
        match_v = torch.allclose(segment_v, expected_v, atol=1e-7)
        print(f"[{'PASS' if match_v else 'FAIL'}] Layer {i}{tag} V weights match")
        all_passed &= match_v

        segment_qb = merged_layer.attention.self.query.bias.data[offset:offset+write_len]
        expected_qb = before[i]['q_b'][:write_len]
        match_qb = torch.allclose(segment_qb, expected_qb, atol=1e-7)
        print(f"[{'PASS' if match_qb else 'FAIL'}] Layer {i}{tag} Q bias match")
        all_passed &= match_qb

        segment_o = merged_layer.attention.output.dense.weight.data[:, offset:offset+write_len]
        expected_o = before[i]['o_w'][:, :write_len]
        match_o = torch.allclose(segment_o, expected_o, atol=1e-7)
        print(f"[{'PASS' if match_o else 'FAIL'}] Layer {i}{tag} W_O columns match")
        all_passed &= match_o

        offset += write_len

    # --- check 3: head_mask_param shape ---
    mask_ok = (merged_layer.head_mask_param.shape[0] == n_head_expected)
    print(f"[{'PASS' if mask_ok else 'FAIL'}] head_mask_param shape: "
          f"expected={n_head_expected}, actual={merged_layer.head_mask_param.shape[0]}")
    all_passed &= mask_ok

    # --- check 4: unrelated layer untouched ---
    untouched_after = test_model.bert.encoder.layer[untouched_idx].attention.self.query.weight.data
    untouched_ok = torch.allclose(untouched_before, untouched_after)
    print(f"[{'PASS' if untouched_ok else 'FAIL'}] Unrelated layer {untouched_idx} unchanged")
    all_passed &= untouched_ok

    # --- check 5: non-target source layers unchanged (they're pruned separately, not modified here) ---
    for i, idx in enumerate(layer_indices):
        if i == target_idx:
            continue
        current_layer = test_model.bert.encoder.layer[idx]
        if current_layer.attention is not None:  # not yet pruned
            src_unchanged = torch.allclose(
                current_layer.attention.self.query.weight.data,
                before[i]['q_w']
            )
            print(f"[{'PASS' if src_unchanged else 'FAIL'}] Source layer {idx} "
                  f"itself unmodified (only target was written to)")
            all_passed &= src_unchanged

    print("="*60)
    print(f"OVERALL: {'ALL TESTS PASSED ✓' if all_passed else 'SOME TESTS FAILED ✗'}")
    print("="*60)

    return all_passed

In [48]:
import math

In [49]:
mha_mlp_contribution = {}

In [50]:
for layers in layer_composition:
    if len(layers) == 1:
        continue

    print(layers)

    head_total_imp = 0
    mlp_scores = []

    # ----------------------------
    # Calculate cluster statistics
    # ----------------------------
    for layer in layers:
        mha_score = sublayer_imp[layer * 2]
        mlp_score = sublayer_imp[layer * 2 + 1]

        head_total_imp += mha_score

        # store layer and its mlp importance
        mlp_scores.append((layer, mlp_score))

    # get top 2 MLP layers
    top_2_mlp_layers = sorted(
        mlp_scores,
        key=lambda x: x[1],
        reverse=True
    )[:2]

    print(head_total_imp)
    print("Top 2 MLP layers:", top_2_mlp_layers)

    obj = {
        "n_heads_per_layer": {},
        "mlp_highest_layers": [x[0] for x in top_2_mlp_layers]
    }

    for layer in layers:
        head_graft_n = math.ceil(
            (12 * sublayer_imp[layer * 2]) / head_total_imp
        )

        print(f'Layer: {layer}, Number of Heads: {head_graft_n}')
        obj['n_heads_per_layer'][layer] = head_graft_n

    mha_mlp_contribution[layers[-1]] = obj

[1, 2]
1.2339795314892967
Top 2 MLP layers: [(2, 0.6468641386632997), (1, 0.46664430442585203)]
Layer: 1, Number of Heads: 7
Layer: 2, Number of Heads: 6
[5, 6, 7]
1.4888206890869444
Top 2 MLP layers: [(5, 0.5560216427947747), (6, 0.4446688402356215)]
Layer: 5, Number of Heads: 9
Layer: 6, Number of Heads: 3
Layer: 7, Number of Heads: 2
[8, 9, 10, 11]
0.7712432476168127
Top 2 MLP layers: [(8, 0.3984632264916888), (9, 0.27978273813072224)]
Layer: 8, Number of Heads: 5
Layer: 9, Number of Heads: 4
Layer: 10, Number of Heads: 3
Layer: 11, Number of Heads: 2


In [51]:
mha_mlp_contribution

{2: {'n_heads_per_layer': {1: 7, 2: 6}, 'mlp_highest_layers': [2, 1]},
 7: {'n_heads_per_layer': {5: 9, 6: 3, 7: 2}, 'mlp_highest_layers': [5, 6]},
 11: {'n_heads_per_layer': {8: 5, 9: 4, 10: 3, 11: 2},
  'mlp_highest_layers': [8, 9]}}

In [52]:
list(mha_mlp_contribution.keys())

[2, 7, 11]

In [53]:
from layer_merge_helper import reorder_layer_neurons, merge_ff

In [54]:
extra_neurons = 2000  # tune this — adds capacity to compensate for dropped layers

for cluster_key in reversed(list(mha_mlp_contribution.keys())):
    cluster           = mha_mlp_contribution[cluster_key]
    layer_indices     = sorted(list(cluster['n_heads_per_layer'].keys()))
    mlp_best_layers   = cluster['mlp_highest_layers']
    target_layer_idx  = layer_indices[0]

    print(f"\n{'='*50}")
    print("Layers in the cluster:", layer_indices)
    print("Best MLP layers:", mlp_best_layers)
    print("Target layer:", target_layer_idx)

    layers      = []
    k_per_layer = []
    for layer_idx in layer_indices:
        layers.append(model.bert.encoder.layer[layer_idx])
        n_head = cluster['n_heads_per_layer'][layer_idx]
        k_per_layer.append(n_head)

    print("k_per_layer:", k_per_layer)

    # --- merge MHA into target (layers[0]) ---
    merged_layer = merge_mha_cluster(model, layers, 0, k_per_layer, torch.device("cpu"))

    # --- merge top-2 FFN layers into target ---
    best_ffn_idx   = mlp_best_layers[0]
    second_ffn_idx = mlp_best_layers[1]

    print(f"\nMerging FFN: source={second_ffn_idx} → target={best_ffn_idx}")
    print(f"  extra_neurons={extra_neurons} → final FFN dim will be "
          f"{max(model.bert.encoder.layer[best_ffn_idx].intermediate.dense.out_features, model.bert.encoder.layer[second_ffn_idx].intermediate.dense.out_features) + extra_neurons}")

    # --- reorder neurons by importance BEFORE merging ---
    print(f"  Reordering neurons in best FFN layer {best_ffn_idx}")
    reorder_layer_neurons(
        layer             = model.bert.encoder.layer[best_ffn_idx],
        neuron_importance = neuron_imp[best_ffn_idx],
        device            = torch.device("cpu")
    )

    print(f"  Reordering neurons in second FFN layer {second_ffn_idx}")
    reorder_layer_neurons(
        layer             = model.bert.encoder.layer[second_ffn_idx],
        neuron_importance = neuron_imp[second_ffn_idx],
        device            = torch.device("cpu")
    )

    # sorted importance vectors for merge_ff allocation loop
    neuron_imp_best   = neuron_imp[best_ffn_idx][
        torch.argsort(neuron_imp[best_ffn_idx], descending=True)
    ]
    neuron_imp_second = neuron_imp[second_ffn_idx][
        torch.argsort(neuron_imp[second_ffn_idx], descending=True)
    ]

    # merge second best FFN into best FFN layer
    merge_ff(
        model,
        sourceLayer           = model.bert.encoder.layer[second_ffn_idx],
        targetLayer           = model.bert.encoder.layer[best_ffn_idx],
        source_int_imp_sorted = neuron_imp_second,
        target_int_imp_sorted = neuron_imp_best,
        device                = torch.device("cpu"),
        extra_neurons         = extra_neurons    # ← now non-zero
    )

    print(f"  Merged FFN dim: {model.bert.encoder.layer[best_ffn_idx].intermediate.dense.out_features}")

    # if merged FFN result is not already in target layer, copy it there
    if best_ffn_idx != target_layer_idx:
        print(f"Copying merged FFN from layer {best_ffn_idx} to target layer {target_layer_idx}")
        target = model.bert.encoder.layer[target_layer_idx]
        source = model.bert.encoder.layer[best_ffn_idx]
        with torch.no_grad():
            target.intermediate.dense = source.intermediate.dense
            target.output.dense       = source.output.dense
            target.output.LayerNorm.weight.data.copy_(source.output.LayerNorm.weight.data.clone())
            target.output.LayerNorm.bias.data.copy_(source.output.LayerNorm.bias.data.clone())
        target.int_mask_param = nn.Parameter(
            torch.ones(
                target.intermediate.dense.out_features,
                dtype=torch.float32,
                device=torch.device("cpu")
            ),
            requires_grad=True
        )
    else:
        print(f"Target layer {target_layer_idx} already has the merged FFN")

    # --- delete source layers (reversed so indices stay valid) ---
    for idx in reversed(range(1, len(layer_indices))):
        layer_idx = layer_indices[idx]
        print(f"Deleting layer {layer_idx} from model")
        del model.bert.encoder.layer[layer_idx]

    print(f"Model now has {len(model.bert.encoder.layer)} layers")


Layers in the cluster: [8, 9, 10, 11]
Best MLP layers: [8, 9]
Target layer: 8
k_per_layer: [5, 4, 3, 2]
Merging cluster of 4 layers into target (idx=0)
  k_per_layer (clamped): [5, 4, 3, 2]
  n_head total: 14, new_dim: 896
  Layer (target): taking 5 heads, head_size=64
  Layer: taking 4 heads, head_size=64
  Layer: taking 3 heads, head_size=64
  Layer: taking 2 heads, head_size=64
    Wrote 5 heads from layer 0 (target) into slots [0:320]
    Wrote 4 heads from layer 1 into slots [320:576]
    Wrote 3 heads from layer 2 into slots [576:768]
    Wrote 2 heads from layer 3 into slots [768:896]
  Done: target now has 14 heads (dim=896)

Merging FFN: source=9 → target=8
  extra_neurons=2000 → final FFN dim will be 5072
  Reordering neurons in best FFN layer 8
  Reordered 3072 neurons
  Reordering neurons in second FFN layer 9
  Reordered 3072 neurons
2073 2999
  Merged FFN dim: 5072
Target layer 8 already has the merged FFN
Deleting layer 11 from model
Deleting layer 10 from model
Deletin

In [55]:
model.to(device)

device

device(type='cuda', index=1)

# 8. Learning Rate and Leanring Scheduler Setup

In [56]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-5)

In [57]:
from train_eval_func import set_lr_scheduler

num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = set_lr_scheduler(
    optimizer          = optimizer,
    num_training_steps = num_training_steps
)

print(num_training_steps)

1150


In [58]:
patience=3

In [59]:
student_layer_ids = [1,2,3,4,5,6]

In [60]:
teacher_layer_ids = []
for layer_idx in layer_composition:
    teacher_layer_ids.append(layer_idx[-1] + 1)

teacher_layer_ids

[1, 3, 4, 5, 8, 12]

In [61]:
layer_composition

[[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]

# 8. Training

In [62]:
from train_eval_func import train

In [63]:
train_stats = train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    num_epochs=num_epochs,
    lr=lr_scheduler,
    patience=patience,
    use_early_stopping=True,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    # save_path=save_path,
    display_epoch_iter=True,
    teacher_model=teacher_model,
    student_layer_ids=student_layer_ids,
    teacher_layer_ids=student_layer_ids
)

Training:   0%|          | 0/1150 [00:00<?, ?it/s]

Knowledge distillation used!


/home/abhinavl/work/Layer_Graft/layer-grafting-publication/scratch_train/../train_eval_func.py:240: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  10%|█         | 115/1150 [00:12<01:48,  9.58it/s]


<----------------- Epoch 1 ----------------->
Loss: 2.72, Training Metrics:
  accuracy: 0.6868
  f1: 0.8029
Validation Loss: 0.88, Validation Metrics:
  accuracy: 0.7304
  f1: 0.8343
Elapsed Time: 14.0581 sec


Training:  20%|██        | 230/1150 [00:27<01:30, 10.20it/s]

...


<----------------- Epoch 5 ----------------->
Loss: 0.46, Training Metrics:
  accuracy: 0.9635
  f1: 0.9730
Validation Loss: 1.45, Validation Metrics:
  accuracy: 0.7966
  f1: 0.8659
Elapsed Time: 14.1977 sec


Training:  60%|█████▉    | 689/1150 [01:23<00:50,  9.17it/s]


<----------------- Epoch 6 ----------------->
Loss: 0.31, Training Metrics:
  accuracy: 0.9736
  f1: 0.9804
Validation Loss: 1.34, Validation Metrics:
  accuracy: 0.7966
  f1: 0.8624
Elapsed Time: 16.9148 sec


Training:  70%|███████   | 805/1150 [01:40<00:34, 10.03it/s]


<----------------- Epoch 7 ----------------->
Loss: 0.26, Training Metrics:
  accuracy: 0.9768
  f1: 0.9828
Validation Loss: 1.49, Validation Metrics:
  accuracy: 0.7990
  f1: 0.8677
Elapsed Time: 14.2072 sec
>>>>> Early stopping callback <<<<<

Total Training Time: 101.67 sec


In [64]:
eval_loop(model, eval_dataloader, task_name, device)

Training:  70%|███████   | 805/1150 [01:41<00:43,  7.92it/s]


({'accuracy': 0.7605797101449275, 'f1': 0.8370808678500986}, 1.670802723478388)

In [65]:
eval_loop(teacher_model, eval_dataloader, task_name, device)

({'accuracy': 0.8133333333333334, 'f1': 0.8686786296900489},
 1.1832497396411512)

In [67]:
eval_loop(model, eval_dataloader, task_name, device)

({'accuracy': 0.8057971014492754, 'f1': 0.8615130219098801},
 1.2663466007862654)